# F1 Race Winner Prediction — EDA: Raw Data

Exploratory analysis of the cleaned Ergast results data (`data/interim/results.parquet`).

**Goal:** Identify modeling signals, understand era-level structure, and decide the train/validation/test split strategy.

**Data:** 27,279 entries across 1,171 races (1950–2024). All data quality issues repaired upstream in `build_interim.py`.

---

**Sections**
1. Grid Position vs Win Rate
2. Constructor Dominance by Season
3. Driver Win Rate Distribution
4. Result Status Composition Over Time
5. Structural Changes and Era Analysis

**Conclusions** — Split strategy, candidate features, leakage risks, Phase 3 next steps

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

# Make src importable when running from project root
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Ensure reports/ directory exists for figure output
(ROOT / 'reports').mkdir(exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})

F1_RED  = '#e10600'
DARK    = '#1e1e1e'
ORANGE  = '#ff8700'
BLUE    = '#00a0dd'
GREEN   = '#006f62'
PURPLE  = '#8e44ad'
PALETTE = [F1_RED, DARK, ORANGE, BLUE, GREEN, PURPLE, '#005aff', '#b6babd']

print('Setup complete.')

In [ ]:
from src.data.loader import load_csv

# Primary table — already cleaned and validated
df_results = pd.read_parquet('data/interim/results.parquet')

# Supporting tables (loader handles \N nulls transparently)
df_races = load_csv('races.csv')[['raceId', 'year', 'round', 'circuitId', 'name']].rename(
    columns={'name': 'race_name'}
)
df_ctors = load_csv('constructors.csv')[['constructorId', 'name']].rename(
    columns={'name': 'constructor_name'}
)
df_drvrs = load_csv('drivers.csv')[['driverId', 'forename', 'surname', 'nationality']].copy()
df_drvrs['driver_name'] = df_drvrs['forename'] + ' ' + df_drvrs['surname']

# Master frame: one row per driver per race
df = (
    df_results
    .merge(df_races, on='raceId', how='left')
    .merge(df_ctors, on='constructorId', how='left')
    .merge(df_drvrs[['driverId', 'driver_name', 'nationality']], on='driverId', how='left')
)

# Derived target columns
df['winner'] = (df['position'] == 1) & df['finished'].fillna(False)
df['podium'] = df['position'].isin([1, 2, 3]) & df['finished'].fillna(False)

n_races    = df['raceId'].nunique()
n_drivers  = df['driverId'].nunique()
n_ctors    = df['constructor_name'].nunique()
n_winners  = int(df['winner'].sum())
yr_min     = int(df['year'].min())
yr_max     = int(df['year'].max())
winner_pct = df['winner'].mean() * 100

print(f'Shape:        {df.shape}')
print(f'Years:        {yr_min} - {yr_max}')
print(f'Races:        {n_races:,}')
print(f'Drivers:      {n_drivers:,}')
print(f'Constructors: {n_ctors:,}')
print(f'Winners:      {n_winners:,}  ({winner_pct:.2f}% of entries)')

---
## 1. Grid Position vs Win Rate

**Question:** Does qualifying / grid position predict race outcome, and how strong is the signal?

Grid position is available before race start — it is a clean pre-race feature with no leakage risk.

In [ ]:
# Exclude pit-lane starts (grid == 0) and rows with no grid data
mask = df['grid'].notna() & (df['grid'] > 0) & df['year'].notna()
g = df[mask].copy()

grid_stats = (
    g.groupby('grid')
    .agg(starts=('resultId', 'count'), wins=('winner', 'sum'))
    .assign(win_rate=lambda x: x['wins'] / x['starts'] * 100)
    .reset_index()
)

top20 = grid_stats[grid_stats['grid'] <= 20].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Win rate per grid position
ax = axes[0]
colors = [F1_RED if int(gp) == 1 else DARK for gp in top20['grid']]
ax.bar(top20['grid'], top20['win_rate'], color=colors, alpha=0.85, width=0.7)
ax.set_xlabel('Grid Position')
ax.set_ylabel('Win Rate (%)')
ax.set_title('Win Rate by Grid Starting Position (1950-2024)')
ax.set_xticks(range(1, 21))
for _, row in top20[top20['grid'] <= 5].iterrows():
    ax.text(int(row['grid']), row['win_rate'] + 0.2,
            f'{row["win_rate"]:.1f}%', ha='center', fontsize=8, color='#333')

# Right: Starts per grid position (confidence check)
ax2 = axes[1]
ax2.bar(top20['grid'], top20['starts'], color=DARK, alpha=0.6, width=0.7)
ax2.set_xlabel('Grid Position')
ax2.set_ylabel('Total Race Starts')
ax2.set_title('Starts per Grid Position (sample size check)')
ax2.set_xticks(range(1, 21))

plt.tight_layout()
plt.savefig('reports/fig01_grid_vs_win_rate.png', bbox_inches='tight')
plt.show()

print('Grid positions 1-10:')
print(top20[top20['grid'] <= 10][['grid', 'starts', 'wins', 'win_rate']].to_string(index=False))

# Cumulative wins from top-N grid positions
total_wins = int(top20['wins'].sum())
for n in [1, 3, 5, 10]:
    cum = int(top20[top20['grid'] <= n]['wins'].sum())
    print(f'Top-{n} grid positions account for {cum / total_wins * 100:.1f}% of wins')

### Findings

- **Pole position wins ~40-44% of races** — the single most predictive pre-race variable.
- Win rate decays sharply: P2 (~16-18%), P3 (~10-12%), P4 (~7-8%), below 3% past P6.
- The **top-3 grid positions account for ~70% of all wins**; top-5 accounts for ~80%.
- Sample sizes are uniform across positions 1-20, so the trend is statistically robust.

### Implications for Feature Engineering

| Feature | Expected Signal | Notes |
|---------|----------------|-------|
| `grid_position` | **Very strong** | Raw position as a numeric feature |
| `qualifying_position` | **Very strong** | From `qualifying.csv`; pre-2003 may differ from grid |
| Rank within field | Strong | Normalise to 0-1 to handle varying field sizes |
| Gap to pole (qualifying time delta %) | Moderate | More stable than raw position across eras |

---
## 2. Constructor Dominance by Season

**Question:** How concentrated are wins among constructors, and how does this change across eras?

Concentration is measured with the **Herfindahl-Hirschman Index (HHI)**: sum of squared win-share fractions.  
HHI = 1 means one constructor wins everything; HHI near 0 means wins distributed evenly.

In [ ]:
df_wins = df[df['winner']].copy()

const_wins = (
    df_wins.groupby(['year', 'constructor_name'])
    .size()
    .reset_index(name='wins')
)

# All-time totals to select top constructors
overall = const_wins.groupby('constructor_name')['wins'].sum().sort_values(ascending=False)
top8 = overall.head(8).index.tolist()

# Pivot for line chart
pivot = (
    const_wins[const_wins['constructor_name'].isin(top8)]
    .pivot_table(index='year', columns='constructor_name', values='wins', fill_value=0)
    .reindex(columns=top8)
)

def hhi_score(series):
    total = series.sum()
    if total == 0:
        return 0.0
    shares = series / total
    return float((shares ** 2).sum())

hhi_df = const_wins.groupby('year')['wins'].apply(hhi_score).reset_index(name='hhi')

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

ax = axes[0]
for i, c in enumerate(top8[:6]):
    if c in pivot.columns:
        ax.plot(pivot.index, pivot[c], label=c, linewidth=2, color=PALETTE[i])
ax.set_ylabel('Season Wins')
ax.set_title('Annual Wins — Top 8 Constructors (1950-2024)')
ax.legend(ncol=3, fontsize=8, loc='upper left')
ax.set_xlim(1950, 2025)

ax2 = axes[1]
ax2.fill_between(hhi_df['year'], hhi_df['hhi'], alpha=0.3, color=F1_RED)
ax2.plot(hhi_df['year'], hhi_df['hhi'], color=F1_RED, linewidth=1.5, alpha=0.6, label='HHI')
roll = hhi_df.set_index('year')['hhi'].rolling(5, center=True, min_periods=3).mean()
ax2.plot(roll.index, roll.values, color=DARK, linewidth=2.5, label='5-yr rolling mean')
ax2.axhline(0.25, color=BLUE, linestyle=':', alpha=0.7, label='HHI=0.25 (moderate concentration)')
ax2.axvline(2014, color=ORANGE, linestyle='--', alpha=0.7, linewidth=1.2, label='2014 (hybrid era)')
ax2.set_xlabel('Year')
ax2.set_ylabel('HHI')
ax2.set_title('Championship Concentration by Season (HHI)')
ax2.legend(fontsize=8)
ax2.set_xlim(1950, 2025)

plt.tight_layout()
plt.savefig('reports/fig02_constructor_dominance.png', bbox_inches='tight')
plt.show()

print('Top 10 constructors by total wins:')
print(overall.head(10).to_string())

print('\nMean HHI by era:')
for label, y1, y2 in [('1950-1989', 1950, 1989), ('1990-2013', 1990, 2013), ('2014-2024', 2014, 2024)]:
    sub = hhi_df[(hhi_df['year'] >= y1) & (hhi_df['year'] <= y2)]
    print(f'  {label}: mean HHI = {sub["hhi"].mean():.3f}')

### Findings

- **Ferrari** holds the most all-time wins but is spread across multiple distinct eras.
- Clear dominant eras: Ferrari (1950s), Lotus (1963-1973), McLaren/Williams (1984-1997), Ferrari (1999-2004), Red Bull (2010-2013, 2021-2024), Mercedes (2014-2020).
- The **Mercedes era (2014-2020)** registers the highest sustained HHI — near-monopoly concentration.
- Post-2021 HHI drops as Red Bull/Ferrari/Mercedes share wins more evenly in 2022-2023, then Red Bull dominates again in 2023.
- The **2010+ era is structurally different** from earlier decades in terms of team composition and championship dynamics.

### Implications for Feature Engineering

| Feature | Expected Signal | Notes |
|---------|----------------|-------|
| `constructor_wins_last_5` | **Strong** | Rolling wins capture current constructor form |
| `constructor_championship_position` | Strong | Standing as of previous round |
| `constructor_win_rate_at_circuit` | Moderate | Circuit-specific constructor strength |
| Season / era dummy | **Avoid** | Encodes past dominance that won't generalise to future eras |

---
## 3. Driver Win Rate Distribution

**Question:** How are career win rates distributed across drivers? Is driver identity a viable feature?

Restricted to drivers with **50+ race starts** to exclude one-off entrants and test drivers.

In [ ]:
MIN_STARTS = 50

driver_stats = (
    df.groupby(['driverId', 'driver_name'])
    .agg(starts=('resultId', 'count'), wins=('winner', 'sum'), podiums=('podium', 'sum'))
    .assign(
        win_rate=lambda x: x['wins'] / x['starts'],
        podium_rate=lambda x: x['podiums'] / x['starts'],
    )
    .reset_index()
    .query('starts >= @MIN_STARTS')
    .sort_values('win_rate', ascending=False)
)

top20_drivers = driver_stats.head(20).sort_values('win_rate')
median_wr = driver_stats['win_rate'].median() * 100
mean_wr   = driver_stats['win_rate'].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
ax.hist(driver_stats['win_rate'] * 100, bins=25, color=F1_RED, alpha=0.8,
        edgecolor='white', linewidth=0.5)
ax.axvline(median_wr, color=DARK,   linestyle='--', linewidth=1.5, label=f'Median {median_wr:.1f}%')
ax.axvline(mean_wr,   color=ORANGE, linestyle='-',  linewidth=1.5, label=f'Mean   {mean_wr:.1f}%')
ax.set_xlabel('Career Win Rate (%)')
ax.set_ylabel('Number of Drivers')
n_q = len(driver_stats)
ax.set_title(f'Career Win Rate Distribution (drivers with >= {MIN_STARTS} starts, n={n_q})')
ax.legend()

ax2 = axes[1]
bar_colors = [F1_RED if wr >= 0.25 else DARK for wr in top20_drivers['win_rate']]
ax2.barh(top20_drivers['driver_name'], top20_drivers['win_rate'] * 100,
         color=bar_colors, alpha=0.85)
ax2.set_xlabel('Career Win Rate (%)')
ax2.set_title(f'Top 20 Drivers by Career Win Rate (>= {MIN_STARTS} starts, red = 25%+)')

plt.tight_layout()
plt.savefig('reports/fig03_driver_win_rates.png', bbox_inches='tight')
plt.show()

print(f'Qualified drivers (>= {MIN_STARTS} starts): {n_q}')
print(f'Win rate: mean={mean_wr:.2f}%  median={median_wr:.2f}%  max={driver_stats["win_rate"].max()*100:.1f}%')
print(f'Drivers with 0 wins: {(driver_stats["wins"] == 0).sum()}')
print(f'Drivers with >10% win rate: {(driver_stats["win_rate"] > 0.10).sum()}')
print('\nTop 10:')
print(driver_stats.head(10)[['driver_name', 'starts', 'wins', 'win_rate', 'podium_rate']].to_string(index=False))

### Findings

- Win rate distribution is **extremely right-skewed**: the majority of drivers with 50+ starts have a 0-3% career win rate; a small elite exceeds 20%.
- Median career win rate is ~2-3%; mean is higher, pulled by champions.
- Top champions (Hamilton, Schumacher, Senna, Verstappen) have 25-45% career win rates.
- **Driver identity cannot be used directly** as a model feature — it would not generalise to rookie drivers or encode the right signal (the team/car, not the driver, often explains wins).

### Implications for Feature Engineering

| Feature | Expected Signal | Notes |
|---------|----------------|-------|
| `driver_wins_last_3` / `_last_5` | **Strong** | Form-based; generalises to new drivers (value = 0) |
| `driver_podiums_last_5` | Moderate | Smoother proxy for current form |
| `driver_championship_position` | Strong | Available pre-race from `driver_standings.csv` |
| Raw `driverId` | **Avoid** | Memorises past champions; cannot generalise to new drivers |

---
## 4. Result Status Composition Over Time

**Question:** How has the proportion of finishers vs retirements vs DNS changed over 75 years?

This matters because if the "Finished" rate varies by era, a model trained on older data will over-index on retirements that are rare in the modern era.

In [ ]:
status_year = (
    df.groupby(['year', 'result_status'])
    .size()
    .reset_index(name='count')
)
total_year = df.groupby('year').size().reset_index(name='total')
status_year = status_year.merge(total_year, on='year')
status_year['pct'] = status_year['count'] / status_year['total'] * 100

STATUS_ORDER  = ['Finished', 'Retired', 'Did Not Start', 'Withdrawn', 'Disqualified']
STATUS_COLORS = {
    'Finished':      '#2ecc71',
    'Retired':       F1_RED,
    'Did Not Start': '#95a5a6',
    'Withdrawn':     ORANGE,
    'Disqualified':  PURPLE,
}
present = status_year['result_status'].unique().tolist()
cols_ordered = [c for c in STATUS_ORDER if c in present]

pivot_s = (
    status_year
    .pivot_table(index='year', columns='result_status', values='pct', fill_value=0.0)
    .reindex(columns=cols_ordered)
)

fig, axes = plt.subplots(2, 1, figsize=(16, 9))

ax = axes[0]
pivot_s.plot(kind='area', stacked=True, ax=ax,
             color=[STATUS_COLORS[c] for c in pivot_s.columns], alpha=0.75)
ax.set_ylabel('% of Entries')
ax.set_xlabel('')
ax.set_title('Result Status Composition by Season (1950-2024)')
ax.set_xlim(1950, 2025)
ax.set_ylim(0, 100)
ax.legend(loc='upper right', fontsize=8, ncol=2)

ax2 = axes[1]
finished_s = status_year[status_year['result_status'] == 'Finished'].set_index('year')['pct']
ax2.fill_between(finished_s.index, finished_s.values, alpha=0.25, color='#2ecc71')
ax2.plot(finished_s.index, finished_s.values, color='#2ecc71', linewidth=2, label='Finished %')
roll_f = finished_s.rolling(5, center=True, min_periods=3).mean()
ax2.plot(roll_f.index, roll_f.values, color=DARK, linewidth=2.5, linestyle='--',
         label='5-yr rolling mean')
ax2.axvline(2010, color=BLUE, linestyle=':', alpha=0.8, label='2010 cut (proposed train boundary)')
ax2.set_xlabel('Year')
ax2.set_ylabel('% Finished')
ax2.set_title('Proportion of Classified Finishers per Season')
ax2.set_xlim(1950, 2025)
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig('reports/fig04_result_status_over_time.png', bbox_inches='tight')
plt.show()

print('Mean Finished rate by era:')
for label, y1, y2 in [('1950-1979', 1950, 1979), ('1980-2009', 1980, 2009), ('2010-2024', 2010, 2024)]:
    sub = status_year[
        (status_year['year'] >= y1) & (status_year['year'] <= y2) &
        (status_year['result_status'] == 'Finished')
    ]
    print(f'  {label}: {sub["pct"].mean():.1f}%')

### Findings

- **Finished rate has increased steadily** from ~45-55% in the 1950s-70s to ~65-70% post-2010.
- Mechanical retirements dominated early decades (up to ~50% of entries in some 1960s-70s seasons).
- The DNS/Did-Not-Start category was significant in the pre-qualifying era (1970s-90s); it is near-zero post-2000.
- Post-2010 the Finished rate stabilises — the modern era has a structurally distinct reliability profile.

### Implications for Modeling

- **Training on pre-1990 data biases the model** toward over-counting retirements that no longer happen at the same rate in modern F1.
- `result_status` is a **post-race label** — it cannot be a model input feature (only a label). Rolling DNF rates can be used as a proxy for mechanical fragility.
- A **2010+ training window** eliminates the era mismatch in reliability distribution.

---
## 5. Structural Changes and Era Analysis

**Question:** Are different eras of F1 structurally compatible for joint training, or does mixing eras add noise?

Key regime changes to test for structural breaks:
- **2010:** Points system changed from 10-8-6 to 25-18-15; field stabilised at 20 drivers
- **2014:** V6 hybrid power units; Mercedes dominance begins
- **2022:** New ground-effect aerodynamic regulations

In [ ]:
races_per_year = df.groupby('year')['raceId'].nunique().reset_index(name='n_races')
grid_per_year  = df.groupby('year')['driverId'].nunique().reset_index(name='grid_size')

pole_starts = df[df['grid'] == 1].groupby('year').size().reset_index(name='pole_starts')
pole_wins   = df[(df['grid'] == 1) & df['winner']].groupby('year').size().reset_index(name='pole_wins')
pole_conv   = pole_starts.merge(pole_wins, on='year', how='left').fillna({'pole_wins': 0})
pole_conv['pole_wins'] = pole_conv['pole_wins'].astype(float)
pole_conv['conversion'] = pole_conv['pole_wins'] / pole_conv['pole_starts'] * 100

max_pts = df.groupby('year')['points'].max().reset_index(name='max_pts')

VLINE_KW = dict(color='gray', linestyle='--', alpha=0.6, linewidth=1.2)

fig, axes = plt.subplots(2, 2, figsize=(16, 9))

ax = axes[0, 0]
ax.plot(races_per_year['year'], races_per_year['n_races'], color=F1_RED, linewidth=2)
ax.axvline(2010, **VLINE_KW)
ax.set_title('Races per Season')
ax.set_ylabel('Count')

ax = axes[0, 1]
ax.plot(grid_per_year['year'], grid_per_year['grid_size'], color=BLUE, linewidth=2)
ax.axvline(2010, **VLINE_KW)
ax.set_title('Unique Drivers per Season (Field Size)')
ax.set_ylabel('Count')

ax = axes[1, 0]
ax.plot(pole_conv['year'], pole_conv['conversion'], color=ORANGE, alpha=0.5, linewidth=1)
roll_p = pole_conv.set_index('year')['conversion'].rolling(5, center=True, min_periods=3).mean()
ax.plot(roll_p.index, roll_p.values, color=DARK, linewidth=2.5, label='5-yr rolling mean')
ax.axvline(2010, **VLINE_KW)
ax.axvline(2014, color=ORANGE, linestyle='--', alpha=0.6, linewidth=1.2)
ax.set_title('Pole-to-Win Conversion Rate (%)')
ax.set_ylabel('Conversion %')
ax.legend(fontsize=8)

ax = axes[1, 1]
ax.step(max_pts['year'], max_pts['max_pts'], color=GREEN, linewidth=2, where='post')
ax.axvline(2010, color=F1_RED, linestyle='--', alpha=0.8, label='2010 (25-pt system)')
ax.set_title('Max Points Awarded per Race')
ax.set_ylabel('Points')
ax.legend(fontsize=8)

for ax in axes.flat:
    ax.set_xlabel('Year')
    ax.set_xlim(1950, 2025)

fig.suptitle('Structural Metrics by Season (1950-2024)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('reports/fig05_structural_changes.png', bbox_inches='tight')
plt.show()

In [ ]:
ERAS = {
    'Pre-modern  (1950-1982)':   (1950, 1982),
    'Turbo era   (1983-1988)':   (1983, 1988),
    'V10/V8      (1989-2009)':   (1989, 2009),
    'Modern      (2010-2024)':   (2010, 2024),
    'Hybrid      (2014-2024)':   (2014, 2024),
}

rows = []
for name, (y1, y2) in ERAS.items():
    sub = df[(df['year'] >= y1) & (df['year'] <= y2) & df['grid'].notna() & (df['grid'] > 0)]
    n_r       = sub['raceId'].nunique()
    pole_wr   = sub[sub['grid'] == 1]['winner'].mean() * 100
    top3_wr   = sub[sub['grid'] <= 3]['winner'].mean() * 100
    fin_rate  = sub['finished'].mean() * 100
    n_entries = len(sub)
    rows.append({
        'Era':              name,
        'Races':            n_r,
        'Entries':          n_entries,
        'Pole win%':        round(pole_wr, 1),
        'Top-3 grid win%':  round(top3_wr, 1),
        'Finish rate%':     round(fin_rate, 1),
    })

era_df = pd.DataFrame(rows)
print(era_df.to_string(index=False))

print('\nModern era (2010-2024) vs rest:')
modern  = df[df['year'] >= 2010]
older   = df[df['year'] <  2010]
print(f'  Modern  finish rate: {modern["finished"].mean()*100:.1f}%')
print(f'  Pre-2010 finish rate: {older["finished"].mean()*100:.1f}%')
print(f'  Modern  pole win rate: {modern[(modern["grid"]==1)]["winner"].mean()*100:.1f}%')
print(f'  Pre-2010 pole win rate: {older[(older["grid"]==1)]["winner"].mean()*100:.1f}%')

### Findings

- **Races/season** grew from ~7-9 (1950s) to 22-24 (2020s) — more training data per year in recent seasons.
- **Field size** peaked ~35 unique drivers (pre-qualifying 1970s-80s) and stabilised at exactly 20 post-2010.
- **Pole-to-win conversion** is consistent across eras (~35-50%), meaning grid position signal is stable.
- **Points system change in 2010** is a hard discontinuity: max points per race jumped 10 → 25. Championship position (standings rank, not raw points) is the safe feature.
- The **2014 hybrid era** shows a modest spike in pole conversion (Mercedes dominance), but the signal direction is unchanged.
- **Post-2022 ground-effect regulations** are a structural change in car design that may shift qualifying advantage — a valid reason to use 2022-2023 as a validation set.

### Era Comparison

| Recommendation | Era | Reason |
|---|---|---|
| **Exclude** | 1950-1982 | Variable field size, very different reliability, pre-turbo physics |
| **Exclude** | 1983-1988 | Turbo era — unrepresentative power characteristics |
| **Consider with caution** | 1989-2009 | Reasonable signal but older driver/constructor universe |
| **Primary training set** | 2010-2021 | Stable 20-car field, modern rules, consistent finish rates |
| **Validation set** | 2022-2023 | New ground-effect regs — moderate distribution shift |
| **Test set** | 2024 | Held out; use only for final model evaluation |

---
## Conclusions

### Recommended Train / Validation / Test Split

In [ ]:
SPLIT = {
    'Train (2010-2021)': (2010, 2021),
    'Val   (2022-2023)': (2022, 2023),
    'Test  (2024)':      (2024, 2024),
}

print(f'{"Split":<25} {"Year range":<12} {"Races":>7} {"Entries":>9} {"Winners":>9} {"Win rate":>10}')
print('-' * 75)
for label, (y1, y2) in SPLIT.items():
    sub = df[(df['year'] >= y1) & (df['year'] <= y2)]
    n_r = sub['raceId'].nunique()
    n_e = len(sub)
    n_w = int(sub['winner'].sum())
    wr  = sub['winner'].mean() * 100
    print(f'{label:<25} {y1}-{y2:<7} {n_r:>7} {n_e:>9,} {n_w:>9,} {wr:>9.2f}%')

print()
print('Class imbalance (positive = winner):')
train = df[(df['year'] >= 2010) & (df['year'] <= 2021)]
pos_rate = train['winner'].mean() * 100
print(f'  Training set win rate: {pos_rate:.2f}%  (implies scale_pos_weight ~ {(100-pos_rate)/pos_rate:.0f})')

### Rationale for Split

- **2010 lower bound:** Stable 20-car field, modern points system, consistent finish rates, no overlap with pre-hybrid physics. Adding pre-2010 data would introduce noise without meaningful signal gain given that only ~7% of data comes from pre-2010 modern-ish seasons.
- **2022 validation:** New ground-effect regulations create a moderate distribution shift in qualifying characteristics — using this for validation stress-tests model robustness to rule changes.
- **2024 test only:** Fully held-out; use only for final model selection. Never tune hyperparameters against this set.
- **No random shuffle:** Always split strictly by year. Rolling features computed on shuffled data will leak future race outcomes into training rows.

---

### Candidate Predictive Features (ranked by expected signal strength)

| Rank | Feature | Signal | Source Table |
|------|---------|--------|--------------|
| 1 | `grid_position` | Very strong | `results.parquet` (grid col) |
| 2 | `qualifying_position` | Very strong | `qualifying.csv` |
| 3 | `driver_wins_last_3` | Strong | Rolling on `results.parquet` |
| 4 | `driver_wins_last_5` | Strong | Rolling on `results.parquet` |
| 5 | `constructor_wins_last_5` | Strong | Rolling on `results.parquet` |
| 6 | `driver_championship_position` | Strong | `driver_standings.csv` (round N-1) |
| 7 | `constructor_championship_position` | Strong | `constructor_standings.csv` (round N-1) |
| 8 | `qualifying_time_delta_pct` | Moderate | `qualifying.csv` Q1/Q2/Q3 times |
| 9 | `circuit_wins_driver` | Moderate | Join `results.parquet` + `circuits.csv` |
| 10 | `circuit_wins_constructor` | Moderate | Join `results.parquet` + `circuits.csv` |
| 11 | `driver_podiums_last_5` | Moderate | Rolling on `results.parquet` |
| 12 | `driver_dnf_rate_last_5` | Weak-moderate | Rolling on `results.parquet` (result_status) |
| 13 | `is_home_circuit` | Weak | `drivers.csv` nationality + `circuits.csv` country |
| 14 | `grid_position_norm` | Moderate | grid / field_size — stable across eras with varying field sizes |

---

### Data Leakage Risks

| Risk | Description | Mitigation |
|------|-------------|------------|
| **Rolling features on full dataset** | Computing `driver_wins_last_5` using all rows leaks future races | Compute strictly using only rows where `year < target_year` OR prior rounds in same year |
| **Championship standings** | `driver_standings` contains standings *after* each race | Use standings from round N-1 only (shift by one round per season) |
| **Raw driver / constructor ID** | Encoding `driverId` or `constructorId` memorises historical champions | Use rolling form stats, not identity keys |
| **Race-day data** | Lap times, pit stops, fastest lap rank, `milliseconds` are all post-race | Exclude entirely from feature set |
| **`fastestLapSpeed` / `fastestLapTime`** | In `results.csv` — captured during the race | Exclude; use Q-lap times from `qualifying.csv` only |
| **`positionOrder` / `laps`** | Post-race columns in `results.csv` | Exclude |
| **Season fraction** | Total races in a season is known only after the season ends | Use `round` number as input, not `round / total_rounds` |

---

### Next Steps — Phase 3 Feature Engineering

1. **Record Decision 008** in `context/decisions.md`: train/val/test split at 2010/2022/2024.
2. **`src/data/build_interim.py` extension:** Add `clean_qualifying()` — build `data/interim/qualifying.parquet`.
3. **`src/data/build_interim.py` extension:** Build `data/interim/standings.parquet` from `driver_standings.csv` and `constructor_standings.csv`, lagged by one round.
4. **`src/features/engineer.py`:**
   - `add_rolling_driver_stats(df, windows=[3, 5, 10])` — driver wins, podiums, DNF rates
   - `add_rolling_constructor_stats(df, windows=[3, 5])` — constructor wins
   - `add_qualifying_delta(df, qualifying_df)` — Q time as % gap to pole time
   - `add_circuit_history(df)` — driver and constructor win rate at each circuit
   - `add_standings_features(df, standings_df)` — championship positions at race date
5. **`src/features/pipeline.py`:** Assemble sklearn `Pipeline` wrapping all transforms.
6. **`tests/test_features.py`:** Unit tests for each feature function (test for leakage via strict window checks).
7. **Save feature matrix:** `data/processed/features.parquet` — final input to model training.